# sqrt-eps-stabilize — faded example 2: Stabilize BatchNorm normalization with eps in denominator

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `sqrt-eps-stabilize`. Running the beacon reports progress on the `Numerical: sqrt-eps stabilization` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Numerical: sqrt-eps stabilization` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`sqrt-eps-stabilize`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "sqrt-eps-stabilize"
DD_SUBTOPIC = "Numerical: sqrt-eps stabilization"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

Batch normalization normalizes each feature across the batch: `(x - mean) / sqrt(var + eps)`. Without eps, any channel where all batch members have the same value (var=0) produces NaN. The standard PyTorch BatchNorm uses eps=1e-5 by default. The eps must be inside the sqrt, not added after, to prevent gradient explosion at small variance.

## Faded exercise 2

Implement `stable_bn_normalize(x, eps=1e-5)` for a (B, C) input tensor. Normalize each channel (column) across the batch dimension.

1. Compute per-channel mean and variance (dim=0).
2. Normalize using `(x - mean) / sqrt(var + eps)`.

The blank step is computing the normalized output with eps inside the sqrt.

**Fill in:** Compute the batch-normalized output by dividing (x - mean) by the square root of (var + eps).

In [ ]:
import torch as t

t.manual_seed(0)

def stable_bn_normalize(x, eps=1e-5):
    mean = x.mean(dim=0)                          # (C,)
    var  = x.var(dim=0, unbiased=False)           # (C,)
    out = None  # TODO: Compute the batch-normalized output by dividing (x - mean) by the square root of (var + eps).
    return out

x = t.tensor([
    [1.0, 7.0, 3.0],
    [3.0, 7.0, 5.0],   # channel 1: constant across batch (var=0)
    [5.0, 7.0, 1.0],
], dtype=t.float32)

out = stable_bn_normalize(x)
print('Output:\n', out)
print('All finite:', t.isfinite(out).all().item())
print('Output shape:', out.shape)


def _test():
    import torch as t

    x = t.tensor([
        [1.0, 7.0, 3.0],
        [3.0, 7.0, 5.0],
        [5.0, 7.0, 1.0],
    ], dtype=t.float32)
    eps = 1e-5
    mean = x.mean(dim=0)
    var  = x.var(dim=0, unbiased=False)
    expected = (x - mean) / t.sqrt(var + eps)
    out = stable_bn_normalize(x, eps)
    assert out.shape == x.shape, f'shape: {out.shape}'
    assert t.allclose(out, expected, atol=1e-5), f'values mismatch'
    assert t.isfinite(out).all(), 'output should be finite (including the constant-column channel)'


try:
    _test()
    _dd_passed.add('faded2')
    print('[Delta Drills] faded2 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t

t.manual_seed(0)

def stable_bn_normalize(x, eps=1e-5):
    mean = x.mean(dim=0)
    var  = x.var(dim=0, unbiased=False)
    out = (x - mean) / t.sqrt(var + eps)
    return out

x = t.tensor([
    [1.0, 7.0, 3.0],
    [3.0, 7.0, 5.0],
    [5.0, 7.0, 1.0],
], dtype=t.float32)

out = stable_bn_normalize(x)
print('Output:\n', out)
print('All finite:', t.isfinite(out).all().item())
print('Output shape:', out.shape)
```
</details>